In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
%run /Workspace/Users/yanquiel@softserve.academy/ecommerce-bronze-platform-/notebooks/utilities

In [0]:
dbutils.widgets.text("catalog", "dbr_dev", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")


catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")



In [0]:
volume_name = "landing"
source_path = f"/Volumes/{catalog}/{bronze_schema}/{volume_name}/{data_source}/*.csv"
bronze_table = f"{catalog}.{bronze_schema}.brz_{data_source}"

In [0]:
schemas = {

    "customers": StructType([
        StructField("customer_id", StringType(), True),
        StructField("customer_unique_id", StringType(), True),
        StructField("customer_zip_code_prefix", IntegerType(), True),
        StructField("customer_city", StringType(), True),
        StructField("customer_state", StringType(), True)
    ]),


    "products": StructType([
        StructField("product_id", StringType(), True),
        StructField("product_category_name", StringType(), True),
        StructField("product_name_lenght", IntegerType(), True),
        StructField("product_description_lenght", IntegerType(), True),
        StructField("product_photos_qty", IntegerType(), True),
        StructField("product_weight_g", IntegerType(), True),
        StructField("product_length_cm", IntegerType(), True),
        StructField("product_height_cm", IntegerType(), True),
        StructField("product_width_cm", IntegerType(), True)
    ]),


    "products_categories": StructType([
        StructField("product_category_name", StringType(), True),
        StructField("product_category_name_english", StringType(), True)
    ])
}



In [0]:
schema = schemas[data_source]

In [0]:
df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(schema)
    .load(source_path)
    .withColumn("source", F.lit(data_source))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
(
    df.write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .mode("overwrite")
    .saveAsTable(bronze_table)
)